# 6. Client-server: jouw database in de cloud

:::{admonition} Leerdoelen
:class: tip

In dit hoofdstuk maak je kennis met **client-server databases**: databases die als aparte server draaien, zodat veel gebruikers er tegelijk mee kunnen werken.

Na dit hoofdstuk kan je:
- uitleggen wat het verschil is tussen een embedded database (SQLite) en een client-server database
- uitleggen waarom bedrijven een databaseserver gebruiken, in eigen beheer (on-premise) of in de cloud
- enkele bekende serverdatabases opnoemen: MySQL, MariaDB, PostgreSQL, SQL Server
- je cafetaria-database uitvoeren op een echte serverdatabase, volledig in de browser
- kleine dialectverschillen tussen SQLite en PostgreSQL herkennen en aanpassen
:::

Tot nu toe werkte je met **SQLite**: in de browser en in DB Browser. Handig, want de hele database is gewoon één bestand op je eigen computer. Maar denk eens terug aan de cafetaria uit het vorige hoofdstuk. Als dat systeem echt gebruikt wordt, bestellen tientallen leerlingen tegelijk vanaf hun smartphone, en moet de kassa diezelfde bestellingen meteen zien binnenkomen. Eén bestand op één laptop volstaat dan niet meer.

In dit hoofdstuk bekijk je hoe dat in de praktijk wordt opgelost — en zet je jouw cafetaria-database over naar een **echte serverdatabase**, zonder ook maar iets te installeren.

## 1 Van bestand naar server

### Hoe SQLite werkt: embedded

Bij SQLite zit de database **in het programma zelf ingebouwd** (Engels: *embedded*). DB Browser opent het bestand `cafetaria.db` rechtstreeks, zoals Word een document opent:

```text
DB Browser  ──leest en schrijft──>  cafetaria.db  (bestand op jouw computer)
```

Dat is snel en eenvoudig, maar het betekent ook: wie met de database wil werken, moet **bij dat bestand** kunnen. Twee kassa's die tegelijk in hetzelfde bestand schrijven? Dat loopt vroeg of laat fout.

### Hoe een client-server database werkt

Bij een **client-server database** draait de database als een **apart programma**: de **server**. Die server staat meestal op een andere computer, en beheert de data volledig zelf. Iedereen die iets wil opvragen of wijzigen — een website, een kassa, een app — is een **client** die via het netwerk een verbinding maakt en SQL naar de server stuurt:

```text
kassa (client)      ──┐
bestelapp (client)  ──┼── netwerk ──>  databaseserver  ──>  data
website (client)    ──┘
```

De server:

* voert de queries uit en stuurt de resultaten terug
* laat **honderden gebruikers tegelijk** werken zonder dat ze elkaar in de weg zitten
* controleert **wie** mag verbinden en **wat** die persoon mag (lezen, wijzigen, verwijderen)

Bekende serverdatabases zijn **MySQL**, **MariaDB**, **PostgreSQL** en **Microsoft SQL Server**. Ze spreken allemaal SQL — dezelfde taal die jij al kent.

### Waar kom je dit tegen?

Bijna overal. Smartschool, een webshop, je bank, ticketverkoop voor een concert: telkens zit er een databaseserver achter waar duizenden clients tegelijk mee verbonden zijn.

Bedrijven hebben daarbij twee opties:

* **on-premise**: het bedrijf zet zelf een server neer en onderhoudt die zelf
* **in de cloud**: het bedrijf huurt een databaseserver in een datacenter bij een cloudleverancier

De cloudoptie wordt steeds populairder: geen eigen hardware, geen eigen onderhoud, en de capaciteit groeit mee met je bedrijf. Ook wij gebruiken vandaag de cloudroute — want een server installeren hoeft dan niet.

## 2 SQLite naast een serverdatabase

| | SQLite | Serverdatabase (bv. PostgreSQL) |
|---|---|---|
| Waar zit de data? | in één bestand op jouw computer | bij de server, vaak in een datacenter |
| Hoe verbind je? | je opent het bestand | via het netwerk, met een adres en een login |
| Meerdere gebruikers tegelijk? | eigenlijk niet | ja, honderden tot duizenden |
| Gebruikersbeheer en rechten? | nee | ja |
| Installatie en onderhoud? | geen | een server die dag en nacht draait |
| Typisch gebruik | apps, smartphones, één gebruiker | websites, bedrijven, scholen |

:::{note}
SQLite is daarom niet "slechter" — het is zelfs de meest gebruikte database ter wereld. Elke smartphone en elke browser bevat er tientallen. De vraag is niet *welke database de beste is*, maar *welke past bij de situatie*: één gebruiker met een lokaal bestand → SQLite; veel gebruikers tegelijk via het netwerk → een serverdatabase.
:::

En het goede nieuws: je databaseontwerp — tabellen, constraints, foreign keys — blijft gewoon geldig. Dat ga je nu zelf bewijzen.

## 3 Aan de slag met DB Fiddle

Een serverdatabase uitproberen kan zonder installatie, met **[DB Fiddle](https://www.db-fiddle.com)**:

* gratis en volledig in de browser — niets te installeren, geen account nodig
* je SQL wordt uitgevoerd op een **echte databaseserver** van DB Fiddle
* je kiest zelf de motor: MySQL of PostgreSQL

Ga naar [db-fiddle.com](https://www.db-fiddle.com) en stel bovenaan links bij **Database** de motor in op **PostgreSQL** (kies de hoogste versie in de lijst).

Je ziet twee invoervakken:

* **Schema SQL** (links): hier zet je de *opbouw* van je database — `CREATE TABLE`-opdrachten en `INSERT`-testdata
* **Query SQL** (rechts): hier zet je de queries die je op die database wil uitvoeren

Met de knop **Run** (of Ctrl+Enter) wordt alles uitgevoerd: eerst het schema, daarna je query. Het resultaat verschijnt onderaan.

:::{note}
DB Fiddle is een **speeltuin** (*playground*): bij elke Run wordt je database op de server **helemaal opnieuw opgebouwd**, en na afloop weer weggegooid. Er blijft dus niets bewaard — perfect om te experimenteren, maar geen plek om een echt systeem op te draaien. In paragraaf 6 zie je hoe het wél permanent kan.
:::

## 4 Jouw cafetaria op een echte server

Je gebruikt nu het cafetaria-ontwerp uit het vorige hoofdstuk — mét constraints en foreign keys. Kan je dat zomaar kopiëren? Bijna. SQL is een standaard, maar elke database spreekt een eigen **dialect**. Voor PostgreSQL zijn er twee kleine verschillen:

1. **Auto-nummering.** In SQLite kreeg een `INTEGER PRIMARY KEY` vanzelf een oplopend nummer. In PostgreSQL schrijf je daarvoor `SERIAL PRIMARY KEY`.
2. **`PRAGMA foreign_keys = ON` valt weg.** Die regel was een eigenaardigheid van SQLite. Een serverdatabase dwingt foreign keys **altijd** af — je kan ze niet eens uitzetten. De `PRAGMA`-regel zou in PostgreSQL zelfs een foutmelding geven: laat ze dus weg.

### Opdracht 1 — Bouw het schema

Zet onderstaande `CREATE TABLE`-opdrachten in het vak **Schema SQL**. Vergelijk met je eigen versie uit het vorige hoofdstuk: behalve `SERIAL` is alles identiek.

```sql
CREATE TABLE leerlingen (
  leerling_id SERIAL PRIMARY KEY,
  naam TEXT NOT NULL CHECK (naam <> ''),
  klas TEXT NOT NULL
);

CREATE TABLE producten (
  product_id SERIAL PRIMARY KEY,
  naam TEXT NOT NULL CHECK (naam <> ''),
  prijs REAL NOT NULL CHECK (prijs >= 0)
);

CREATE TABLE bestellingen (
  bestelling_id SERIAL PRIMARY KEY,
  datum TEXT NOT NULL,
  status TEXT NOT NULL CHECK (status IN ('open', 'betaald', 'geannuleerd')),
  leerling_id INTEGER NOT NULL,
  FOREIGN KEY (leerling_id)
    REFERENCES leerlingen(leerling_id)
    ON DELETE RESTRICT
);

CREATE TABLE bestelling_lijnen (
  bestellijn_id SERIAL PRIMARY KEY,
  bestelling_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  aantal INTEGER NOT NULL CHECK (aantal > 0),
  FOREIGN KEY (bestelling_id)
    REFERENCES bestellingen(bestelling_id)
    ON DELETE CASCADE,
  FOREIGN KEY (product_id)
    REFERENCES producten(product_id)
    ON DELETE RESTRICT
);
```

### Opdracht 2 — Voeg geldige testdata toe

Zet daaronder, nog steeds in **Schema SQL**, de geldige testdata:

```sql
INSERT INTO leerlingen (naam, klas) VALUES
('Aya El Mounir', '5BW'),
('Noah Peeters', '5BW'),
('Louise Janssens', '5AW');

INSERT INTO producten (naam, prijs) VALUES
('Kaasbroodje', 2.50),
('Bicky burger', 4.20),
('Water', 1.00);

INSERT INTO bestellingen (datum, status, leerling_id) VALUES
('2026-02-02', 'open', 1),
('2026-02-02', 'betaald', 2);

INSERT INTO bestelling_lijnen (bestelling_id, product_id, aantal) VALUES
(1, 1, 1),
(1, 3, 2),
(2, 2, 1);
```

### Opdracht 3 — Het bonnetje, maar dan op een server

Zet in het vak **Query SQL** de bonnetjes-query uit het vorige hoofdstuk en klik op **Run**:

```sql
SELECT
  bestellingen.bestelling_id,
  leerlingen.naam AS leerling,
  producten.naam AS product,
  bestelling_lijnen.aantal,
  producten.prijs,
  (bestelling_lijnen.aantal * producten.prijs) AS lijn_totaal
FROM bestelling_lijnen
LEFT JOIN bestellingen ON bestellingen.bestelling_id = bestelling_lijnen.bestelling_id
LEFT JOIN leerlingen   ON leerlingen.leerling_id = bestellingen.leerling_id
LEFT JOIN producten    ON producten.product_id = bestelling_lijnen.product_id
ORDER BY bestellingen.bestelling_id, bestelling_lijnen.bestellijn_id;
```

**Wat moet je zien?**

Exact hetzelfde bonnetje als in DB Browser. Zelfde SQL, zelfde resultaat — maar dit keer is de query via het internet naar een PostgreSQL-server gestuurd, daar uitgevoerd, en kwam alleen het resultaat naar jouw browser terug. Jouw laptop was enkel de **client**.

## 5 Houdt de server ook fouten tegen?

In het vorige hoofdstuk hield SQLite foute gegevens tegen dankzij je constraints en foreign keys. Doet de server dat ook?

### Opdracht 4 — Probeer de foute INSERTs

Vervang de inhoud van **Query SQL** telkens door één van onderstaande regels en klik op **Run**. Noteer per regel welke foutmelding je krijgt.

```sql
INSERT INTO producten (naam, prijs) VALUES ('Chocomelk', -1.50);
```

```sql
INSERT INTO bestellingen (datum, status, leerling_id) VALUES ('2026-02-02', 'klaar', 3);
```

```sql
INSERT INTO bestelling_lijnen (bestelling_id, product_id, aantal) VALUES (2, 1, 0);
```

```sql
INSERT INTO bestellingen (datum, status, leerling_id) VALUES ('2026-02-02', 'open', 999);
```

```sql
INSERT INTO bestelling_lijnen (bestelling_id, product_id, aantal) VALUES (99, 1, 1);
```

**Wat moet je zien?**

* De eerste drie falen met `violates check constraint`: een **constraint** grijpt in (negatieve prijs, onbestaande status, aantal 0).
* De laatste twee falen met `violates foreign key constraint`: een **foreign key** grijpt in (leerling 999 en bestelling 99 bestaan niet).

Elke regel wordt dus geweigerd — door de server zelf, nog vóór er iets in de tabellen terechtkomt.

### Opdracht 5 — ON DELETE op de server

Test ook het verwijdergedrag. Zet in **Query SQL**:

```sql
DELETE FROM bestellingen WHERE bestelling_id = 1;
SELECT * FROM bestelling_lijnen;
```

➡️ De bestellijnen van bestelling 1 zijn mee verdwenen: **ON DELETE CASCADE**.

Probeer daarna:

```sql
DELETE FROM producten WHERE product_id = 2;
```

➡️ Dit wordt geweigerd, want de Bicky burger zit nog in een bestelling: **ON DELETE RESTRICT**.

💡 Niets kapotgemaakt: bij de volgende Run bouwt DB Fiddle je database gewoon opnieuw op vanuit het Schema SQL-vak.

### Controlevragen

1. Waarom moest je `PRAGMA foreign_keys = ON` hier nergens gebruiken?
2. Je kreeg op de server dezelfde weigeringen als in DB Browser. Wat zegt dat over je **ontwerp**? Hangt een goed ontwerp af van het databasesysteem?
3. Wat gebeurt er met je tabellen en data telkens je op Run klikt? Waarom is DB Fiddle daardoor geen bruikbare oplossing voor de echte cafetaria?
4. Som twee dingen op die een serverdatabase kan en SQLite niet.

## 6 Verder dan de speeltuin

DB Fiddle laat je proeven van een echte server, maar een speeltuin heeft grenzen: er wordt niets bewaard, en er zijn geen gebruikers of rechten. Voor een echt systeem — de cafetaria die volgend schooljaar écht draait — huur je een **permanente clouddatabase**.

Dat kan gratis bij **[Supabase](https://supabase.com)**: een cloudplatform dat voor jou een volwaardige PostgreSQL-server opzet.

* gratis account, **zonder kredietkaart** (tot 2 gratis projecten van elk 500 MB)
* een **SQL Editor** in de browser, plus een **Table Editor** waarin je je tabellen en data ziet zoals in DB Browser
* je project **blijft bestaan** tussen de lessen — wat je vandaag bouwt, staat er volgende week nog

⚠️ Een gratis project wordt **gepauzeerd** na een week zonder activiteit. Je data blijft bewaard; in het dashboard maak je het project met één klik weer wakker.

### Wil je het proberen? (thuis, optioneel)

1. Maak een gratis account op [supabase.com](https://supabase.com) en klik op **New project**.
2. Open links de **SQL Editor**, plak het schema en de testdata uit paragraaf 4, en voer ze uit.
3. Voer de bonnetjes-query uit — en bekijk je tabellen ook eens in de **Table Editor**.
4. Kom morgen terug: alles staat er nog. Jouw database draait nu echt in de cloud.

💡 Ook [sqliteonline.com](https://sqliteonline.com) biedt naast SQLite sessies aan op MariaDB-, PostgreSQL- en SQL Server-servers — handig om nog eens een ander dialect te zien.

## Einde van hoofdstuk 6 — Wat je nu zou moeten kunnen

* het verschil uitleggen tussen een embedded database en een client-server database
* uitleggen wat een client en een server zijn, en waarom veel gebruikers tegelijk een server nodig maken
* de opties on-premise en cloud benoemen, met hun voor- en nadelen
* enkele serverdatabases opnoemen: MySQL, MariaDB, PostgreSQL, SQL Server
* een ontwerp met constraints en foreign keys uitvoeren op een PostgreSQL-server in de browser
* dialectverschillen herkennen: `SERIAL` in plaats van auto-nummering via `INTEGER PRIMARY KEY`, en geen `PRAGMA` meer

Daarmee is de cirkel rond: je hebt een database **ontworpen** (ERD), **beveiligd** (constraints en foreign keys) en **uitgevoerd op een echte server** — precies zoals dat in bedrijven gebeurt.